# Export Analytical Model using Autodesk Automation API

This notebook demonstrates how to export a Revit structural analytical model to JSON using the Autodesk Platform Services (APS) Design Automation API with Revit 2025 (.NET 8).

The workflow uploads a Revit model to OSS (Object Storage Service), runs the `AnalyticalExportDA` app bundle via a Design Automation work item, and downloads the resulting `analytical_export.json`.

**Note:** The input Revit file and compiled app bundle (ZIP) can be found in `2025 - Net8/files`

## Step 1: Setup and Import Libraries

Import all necessary libraries and modules.

In [ ]:
import os
import json
import logging
import uuid
from pathlib import Path
from dotenv import load_dotenv

from aps_automation_sdk.classes import (
    Activity,
    ActivityInputParameter,
    ActivityOutputParameter,
    AppBundle,
    WorkItem
)

from aps_automation_sdk.utils import (
    delete_activity,
    delete_appbundle,
    get_token,
    set_nickname
)

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

## Step 2: Load Environment Variables and Configure Authentication

Load credentials from environment variables and get an authentication token.

**Prerequisites:**
1. Create an APS application to get your credentials. Follow these tutorials:
   - Create App: https://aps.autodesk.com/en/docs/oauth/v2/tutorials/create-app
   - Design Automation Setup: https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step2-create-forge-app/

2. Create a `.env` file in the project root with your credentials:
   ```
   CLIENT_ID=your_client_id
   CLIENT_SECRET=your_client_secret
   ```

**Note on Nicknames:** The nickname must be unique across all APS apps globally. If you encounter errors, try using alphanumeric unique values. Choose a distinctive nickname to avoid conflicts.

In [ ]:
load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID", "")
CLIENT_SECRET = os.getenv("CLIENT_SECRET", "")

# Get authentication token
token = get_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)

# Set nickname. If the app already has a nickname, the previous one will be returned.
nickname = set_nickname(token, "myUniqueNickNameHere")

print(f"Authentication successful. Nickname: {nickname}")

## Step 3: Define Project Constants

Set up all the constants needed for the analytical model export workflow.

**Note:** The input Revit file and compiled app bundle ZIP can be found in `2025 - Net8/files`. Build the bundle first by running `build-da-bundle.ps1`.

In [ ]:
YEAR = 2025

# Define constants
app_bundle_name = f"AnalyticalExportDA{YEAR}"
activity_name = f"AnalyticalExportActivity{YEAR}"
alias = "dev"
bucket_key = uuid.uuid4().hex

zip_path = Path.cwd() / "2025 - Net8" / "files" / "AnalyticalExportDA.bundle.zip"
input_rvt_path = Path.cwd() / "2025 - Net8" / "files" / "StructuralTemplate.rvt"
output_json_path = Path.cwd() / "2025 - Net8" / "files" / "output" / "analytical_export.json"

# Create full aliases
appbundle_full_alias = f"{nickname}.{app_bundle_name}+{alias}"
activity_full_alias = f"{nickname}.{activity_name}+{alias}"

print(f"App Bundle: {appbundle_full_alias}")
print(f"Activity:   {activity_full_alias}")
print(f"Bundle ZIP: {zip_path}")
print(f"Input RVT:  {input_rvt_path}")

## Step 4: Create and Deploy App Bundle

Register and upload the app bundle that contains the analytical model export logic.

**Note:** This wrapper covers all the API calls from the tutorial step: https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step4-publish-appbundle/

In [ ]:
bundle = AppBundle(
    appBundleId=app_bundle_name,
    engine=f"Autodesk.Revit+{YEAR}",
    alias=alias,
    zip_path=str(zip_path),
    description=f"Analytical export for Revit {YEAR}",
)

bundle.deploy(token)
print("App bundle deployed successfully!")

## Step 5: Define Activity Parameters

Create input and output parameters for the activity.

- **inputModel** – The Revit `.rvt` file to process (engine input).
- **exportJson** – The resulting `analytical_export.json` written by the addin.

**Reference:** https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step5-publish-activity/#step-1-create-a-new-activity

In [ ]:
# Input: Revit model file
input_revit = ActivityInputParameter(
    name="inputModel",
    localName="input.rvt",
    verb="get",
    description="Input Revit model",
    required=True,
    is_engine_input=True,
    bucketKey=bucket_key,
    objectKey="input.rvt",
)

# Output: Exported analytical model JSON
output_result = ActivityOutputParameter(
    name="exportJson",
    localName="analytical_export.json",
    verb="put",
    description="Exported analytical model JSON",
    bucketKey=bucket_key,
    objectKey="analytical_export.json",
)

print("Activity parameters defined successfully!")

## Step 6: Create and Deploy Activity

Create the activity that links the app bundle with the parameters.

**Note:** This wrapper covers all the API calls from the tutorial step: https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step5-publish-activity/

In [ ]:
activity = Activity(
    id=activity_name,
    parameters=[input_revit, output_result],
    engine=f"Autodesk.Revit+{YEAR}",
    appbundle_full_name=appbundle_full_alias,
    description=f"Export analytical model JSON for Revit {YEAR}",
    alias=alias,
)

# Set the Revit command line
activity.set_revit_command_line()
activity.deploy(token=token)
print("Activity deployed successfully!")

## Step 7: Upload Input Revit File

Upload the Revit model to Object Storage Service (OSS).

**Note:** This wrapper covers all the API calls from the tutorial step: https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step6-prepare-cloud-storage/

In [ ]:
input_revit.upload_file_to_oss(file_path=str(input_rvt_path), token=token)
print(f"Input Revit file uploaded: {input_rvt_path.name}")

## Step 8: Create and Execute Work Item

Create a work item with all parameters and submit it to Design Automation.

**Note:** This wrapper covers all the API calls from the tutorial step: https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step7-post-workitem/

In [ ]:
work_item = WorkItem(
    parameters=[input_revit, output_result],
    activity_full_alias=activity_full_alias,
)

print("Work item created. Starting execution...")

## Step 9: Poll Work Item Status

Monitor the work item execution until completion (max 15 minutes).

In [ ]:
status_resp = work_item.execute(token=token, max_wait=900, interval=10)
last_status = status_resp.get("status", "")

print(f"Work item completed with status: {last_status}")
print(json.dumps(status_resp, indent=2))

## Step 10: Download Results

If the work item succeeded, download the exported analytical model JSON.

**Note:** This wrapper covers all the API calls from the tutorial step: https://aps.autodesk.com/en/docs/design-automation/v3/tutorials/revit/step7-post-workitem/

In [ ]:
if last_status == "success":
    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    output_result.download_to(output_path=str(output_json_path), token=token)
    print(f"✅ Download successful: {output_json_path}")
else:
    print(f"❌ Work item failed or did not complete successfully. Status: {last_status}")

## Step 11: Cleanup Resources

Delete the activity and app bundle to clean up resources.

In [ ]:
print("Cleaning up: deleting activity and app bundle...")

try:
    delete_activity(activityId=activity_name, token=token)
    print(f"✅ Activity deleted: {activity_full_alias}")
except Exception as e:
    print(f"⚠️ Failed to delete activity: {e}")

try:
    delete_appbundle(appbundleId=app_bundle_name, token=token)
    print(f"✅ App bundle deleted: {app_bundle_name}")
except Exception as e:
    print(f"⚠️ Failed to delete app bundle: {e}")

print("\n🎉 Analytical model export workflow completed!")